# Backpropagation Demystified: Neural Nets from First Principles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/backpropagation.ipynb)

Build a neural network from scratch and understand backpropagation through the chain rule.

**Blog post:** [sesen.ai/blog/backpropagation-neural-nets-from-first-principles](https://sesen.ai/blog/backpropagation-neural-nets-from-first-principles)

**What you'll learn:**
- How gradient descent optimises parameters
- The forward pass and backward pass of a neural network
- How the chain rule enables backpropagation
- Why XOR requires a hidden layer

**Prerequisites:** Basic Python and NumPy. Some calculus (derivatives, chain rule) is helpful but not required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## Part 1: Gradient Descent on a Line

Before building a neural network, let's see gradient descent in its simplest form: fitting $y = ax + b$ to noisy data.

The idea: start with random `a` and `b`, then repeatedly nudge them in the direction that reduces the mean squared error.

In [ ]:
# Generate noisy data: y = 2x + 1
np.random.seed(42)
x_data = np.linspace(0, 2, 30)
y_data = 2 * x_data + 1 + np.random.normal(0, 0.3, 30)

# Start with random parameters
a, b = 0.0, 0.0
lr = 0.1
history = [(a, b)]

# Gradient descent: 80 steps
for _ in range(80):
    y_pred = a * x_data + b
    error = y_pred - y_data
    grad_a = (2 / len(x_data)) * np.sum(error * x_data)
    grad_b = (2 / len(x_data)) * np.sum(error)
    a -= lr * grad_a
    b -= lr * grad_b
    history.append((a, b))

print(f"Final: a={a:.4f}, b={b:.4f}")
print(f"True:  a=2.0000, b=1.0000")
print(f"Final MSE: {np.mean((a * x_data + b - y_data) ** 2):.6f}")

In [ ]:
# Animate gradient descent
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_data, y_data, alpha=0.6, color='#2563eb', label='Data', zorder=3)
line, = ax.plot([], [], 'r-', linewidth=2.5, zorder=4)
title_text = ax.set_title('', fontsize=13, fontweight='bold')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_xlim(-0.1, 2.1)
ax.set_ylim(-0.5, 6)
ax.grid(True, alpha=0.3)

def update(frame):
    a_f, b_f = history[frame]
    line.set_data(x_data, a_f * x_data + b_f)
    loss = np.mean((a_f * x_data + b_f - y_data) ** 2)
    title_text.set_text(f'Step {frame}: a={a_f:.2f}, b={b_f:.2f}, MSE={loss:.3f}')
    return line, title_text

anim = FuncAnimation(fig, update, frames=len(history), interval=100, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

The line starts flat and wobbles into place. At each step we compute:

$$\frac{\partial L}{\partial a} = \frac{2}{n} \sum_i (\hat{y}_i - y_i) \cdot x_i \qquad \frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)$$

With just 2 parameters, computing gradients is trivial. But a neural network has thousands of parameters arranged in layers. How do we compute all the gradients efficiently? That's what **backpropagation** solves.

## Part 2: Building a Neural Network from Scratch

### The XOR Problem

XOR (exclusive or) returns 1 when inputs differ, 0 when they're the same:

| Input 1 | Input 2 | XOR |
|---------|---------|-----|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

No straight line can separate the 1s from the 0s. This is why Minsky & Papert (1969) argued that single-layer perceptrons were fundamentally limited. The fix: add a **hidden layer**.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(output):
    """Derivative of sigmoid given its output (not its input)."""
    return output * (1 - output)

In [ ]:
class NeuralNetwork:
    def __init__(self, layer_sizes, learning_rate=0.5):
        self.lr = learning_rate
        self.weights = []
        self.biases = []
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i + 1]) * 0.5
            b = np.random.randn(1, layer_sizes[i + 1]) * 0.5
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, X):
        """Forward pass: compute output and cache activations."""
        self.activations = [X]
        current = X
        for w, b in zip(self.weights, self.biases):
            current = sigmoid(current @ w + b)
            self.activations.append(current)
        return current

    def backward(self, y_true):
        """Backward pass: compute gradients and update weights."""
        m = y_true.shape[0]
        # Output layer error
        delta = (self.activations[-1] - y_true) * sigmoid_derivative(self.activations[-1])

        # Propagate backwards through layers
        for i in range(len(self.weights) - 1, -1, -1):
            grad_w = self.activations[i].T @ delta / m
            grad_b = np.sum(delta, axis=0, keepdims=True) / m
            if i > 0:
                delta = (delta @ self.weights[i].T) * sigmoid_derivative(self.activations[i])
            self.weights[i] -= self.lr * grad_w
            self.biases[i] -= self.lr * grad_b

    def train(self, X, y, epochs=10000):
        """Train and return loss history."""
        losses = []
        for epoch in range(epochs):
            output = self.forward(X)
            loss = np.mean((y - output) ** 2)
            losses.append(loss)
            self.backward(y)
        return losses

In [ ]:
# XOR data
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

# Train: 2 inputs -> 4 hidden -> 1 output
np.random.seed(42)
nn = NeuralNetwork([2, 4, 1], learning_rate=2.0)
losses = nn.train(X, y, epochs=10000)

# Results
print("Predictions after training:")
for inputs, target in zip(X, y):
    pred = nn.forward(inputs.reshape(1, -1))
    print(f"  {inputs} -> {pred[0, 0]:.4f} (target: {target[0]})")

The network learns XOR perfectly! Values near 0 for XOR=0, near 1 for XOR=1.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Loss curve
ax1.plot(losses, color='#2563eb', linewidth=1.5)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('MSE Loss', fontsize=12)
ax1.set_title('Training Loss', fontsize=13, fontweight='bold')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200),
                      np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = nn.forward(grid).reshape(xx.shape)

ax2.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.8)
ax2.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap='RdBu_r',
            edgecolors='black', s=200, linewidths=2, zorder=5)
for i, (xi, yi) in enumerate(X):
    ax2.annotate(f'XOR={y[i,0]}', (xi, yi), textcoords='offset points',
                xytext=(12, 8), fontsize=10, fontweight='bold')
ax2.set_xlabel('Input 1', fontsize=12)
ax2.set_ylabel('Input 2', fontsize=12)
ax2.set_title('XOR Decision Boundary', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## Part 3: Understanding the Backward Pass

### The Chain Rule

For a single output weight $w_{jk}$ connecting hidden neuron $j$ to output neuron $k$:

$$\frac{\partial L}{\partial w_{jk}} = \frac{\partial L}{\partial \hat{y}_k} \cdot \frac{\partial \hat{y}_k}{\partial z_k} \cdot \frac{\partial z_k}{\partial w_{jk}}$$

Where:
- $\partial L / \partial \hat{y}_k = \hat{y}_k - y_k$ — how wrong is the prediction?
- $\partial \hat{y}_k / \partial z_k = \hat{y}_k(1 - \hat{y}_k)$ — sigmoid derivative
- $\partial z_k / \partial w_{jk} = h_j$ — the hidden activation

For hidden layers, the chain extends further — the error from the output propagates back through the weights.

### Tracing One Weight Update

Let's manually trace the gradient for a single weight to verify our code:

In [ ]:
# Create a fresh network with known weights
np.random.seed(42)
nn_trace = NeuralNetwork([2, 4, 1], learning_rate=2.0)

# Single forward pass with input [0, 1] (target: 1)
test_input = np.array([[0, 1]])
test_target = np.array([[1]])

output = nn_trace.forward(test_input)
print(f"Prediction: {output[0, 0]:.6f}")
print(f"Target: 1.0")
print(f"Error (output - target): {output[0, 0] - 1:.6f}")
print()

# Manual gradient computation for the last weight (weights[1][0, 0])
y_hat = output[0, 0]
h = nn_trace.activations[1][0, 0]  # hidden activation

# Chain rule:
dL_dy = y_hat - 1.0                          # dL/dy_hat
dy_dz = y_hat * (1 - y_hat)                  # sigmoid derivative
dz_dw = h                                     # hidden activation

manual_gradient = dL_dy * dy_dz * dz_dw
print(f"Manual gradient for weights[1][0,0]: {manual_gradient:.6f}")

# Compare with code's gradient
delta = (output - test_target) * sigmoid_derivative(output)
code_gradient = (nn_trace.activations[1].T @ delta)[0, 0]
print(f"Code's gradient for weights[1][0,0]:  {code_gradient:.6f}")
print(f"Match: {np.isclose(manual_gradient, code_gradient)}")

## Part 4: Experimenting

### Effect of Learning Rate

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for ax, lr_val in zip(axes, [0.01, 0.5, 2.0, 10.0]):
    np.random.seed(42)
    nn_exp = NeuralNetwork([2, 4, 1], learning_rate=lr_val)
    exp_losses = nn_exp.train(X, y, epochs=5000)
    ax.plot(exp_losses, linewidth=1.5)
    ax.set_title(f'lr = {lr_val}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_yscale('log')
    ax.set_ylim(1e-6, 1)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('MSE Loss')
fig.suptitle('Effect of Learning Rate on XOR Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Effect of Network Architecture

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
architectures = [
    ([2, 2, 1], '2-2-1 (minimal)'),
    ([2, 4, 1], '2-4-1 (standard)'),
    ([2, 4, 4, 1], '2-4-4-1 (deep)'),
]

for ax, (arch, name) in zip(axes, architectures):
    np.random.seed(42)
    nn_arch = NeuralNetwork(arch, learning_rate=2.0)
    arch_losses = nn_arch.train(X, y, epochs=10000)
    ax.plot(arch_losses, linewidth=1.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

    # Print final accuracy
    preds = [nn_arch.forward(x.reshape(1, -1))[0, 0] for x in X]
    print(f"{name}: final loss={arch_losses[-1]:.6f}, preds={[f'{p:.3f}' for p in preds]}")

axes[0].set_ylabel('MSE Loss')
fig.suptitle('Effect of Architecture on XOR Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 5: The Original Code

Our implementation is a cleaned-up, vectorised version of the classic backpropagation example from [Matt Mazur's blog](https://mattmazur.com/2015/03/17/a-step-by-step-backpropagation-example/). Here's the original-style approach with explicit neuron objects, using the same fixed weights from the blog post:

In [ ]:
# Matt Mazur's blog post example with fixed initial weights
np.random.seed(42)
nn_mazur = NeuralNetwork([2, 2, 2], learning_rate=0.5)

# Set weights to match the blog post
nn_mazur.weights[0] = np.array([[0.15, 0.25], [0.20, 0.30]])
nn_mazur.biases[0] = np.array([[0.35, 0.35]])
nn_mazur.weights[1] = np.array([[0.40, 0.50], [0.45, 0.55]])
nn_mazur.biases[1] = np.array([[0.60, 0.60]])

# Train on the blog post's single example
mazur_X = np.array([[0.05, 0.1]])
mazur_y = np.array([[0.01, 0.99]])

mazur_losses = []
for i in range(10000):
    out = nn_mazur.forward(mazur_X)
    loss = np.mean((mazur_y - out) ** 2)
    mazur_losses.append(loss)
    nn_mazur.backward(mazur_y)

print(f"Initial loss: {mazur_losses[0]:.6f}")
print(f"Final loss:   {mazur_losses[-1]:.9f}")
print(f"Final output: {nn_mazur.forward(mazur_X)}")
print(f"Target:       {mazur_y}")

## Exercises

1. **Trace a weight update** — Pick one specific weight in the XOR network. Compute its gradient by hand for one training example and verify it matches the code.

2. **Learning rate experiments** — Try rates of 0.01, 0.5, 2.0, 10.0 (already done above). At what learning rate does training diverge?

3. **Add a layer** — Change the architecture from [2, 4, 1] to [2, 4, 4, 1]. Does it learn XOR faster or slower? Why?

4. **ReLU activation** — Replace sigmoid with `max(0, x)`. You'll need to change `sigmoid_derivative` too (it becomes 1 if x > 0, else 0). What changes in the training dynamics?

5. **Batch vs online learning** — Modify the code to update weights after each individual example instead of all four. Compare convergence speed.

6. **Numerical gradient checking** — Compute the gradient numerically: $\frac{\partial L}{\partial w} \approx \frac{L(w + \epsilon) - L(w - \epsilon)}{2\epsilon}$ for small $\epsilon$. Verify it matches the backprop gradient.

## References

- Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature*, 323(6088), 533-536.
- Mazur, M. (2015). A Step by Step Backpropagation Example. https://mattmazur.com/2015/03/17/a-step-by-step-backpropagation-example/
- Linnainmaa, S. (1970). The representation of the cumulative rounding error of an algorithm as a Taylor expansion of the local rounding errors. *Master's thesis*, University of Helsinki.
- Werbos, P. J. (1974). Beyond Regression: New Tools for Prediction and Analysis in the Behavioral Sciences. *PhD thesis*, Harvard University.